# The Transformation Logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER(ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_key AS customer_number,
    ci.customer_firstname AS first_name,
    ci.customer_lastname AS last_name,
    la.country,
    ci.customer_marital_status AS marital_status,
    CASE 
        WHEN ci.customer_gender <> 'n/a' THEN ci.customer_gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.created_at AS create_date
FROM silver.crm_customers AS ci
LEFT JOIN silver.erp_customers AS ca
    ON ci.customer_key = ca.customer_number
LEFT JOIN silver.erp_customer_location AS la
    ON ci.customer_key = la.customer_number;
    """

df = spark.sql(query)

In [0]:
df.limit(10).display()

## Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

## Sanity checks of Gold table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10